# 05 - Dataset Design & Quality Control

This notebook inspects the **1,000-run V3 dataset** used to train and evaluate all ML surrogates.
Rather than regenerating the dataset, we load pre-computed QC reports and dataset statistics
from `results/dataset/`.

**Design choices** (see `configs/sim/v3_literature_dataset_spec.yaml`):

| Parameter | Sampling | Range / Values |
|-----------|----------|----------------|
| `patch_width` | Discrete | 0.25, 0.5, 1.0 cm |
| `patch_offset` | Discrete | left, center, right |
| `C_0` | Uniform | [0.8, 1.2] |
| `decay_rate` | Uniform | [0.05, 0.3] s^-1 |
| `k_dermis` | Discrete | 0, 1e-5, 3.3e-5, 1e-4 s^-1 |
| `heterogeneity_sigma` | Uniform | [0.01, 0.08] |
| `heterogeneity_steps` | Random int | [3, 9] |

**Splits:** 700 train / 150 val / 150 test (seed 321, stratified by `patch_width`).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from skin_diffusion.notebook_helpers import setup_repo_root

repo_root = setup_repo_root()

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'font.size': 10,
    'legend.fontsize': 9,
})

# Load pre-computed results
qc = json.load(open('results/dataset/qc_report.json'))
meta = json.load(open('results/dataset/meta.json'))
print('QC report and dataset meta loaded.')
print(f'Splits: {meta["splits"]}')
print(f'Features ({len(meta["feature_names"])}): {meta["feature_names"]}')
print(f'Targets: {meta["scalar_target_names"]}')

## 1 - Split counts & overlap check

In [ ]:
counts = qc['counts']
splits = ['train', 'val', 'test']
sizes = [counts[s] for s in splits]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig.suptitle('Dataset composition', fontsize=14, fontweight='bold')

# -- Pie chart --
colours = ['#3498db', '#f39c12', '#e74c3c']
wedges, texts, autotexts = ax1.pie(
    sizes, labels=[f'{s}\n({n})' for s, n in zip(splits, sizes)],
    colors=colours, autopct='%1.0f%%', startangle=90,
    textprops={'fontsize': 11})
ax1.set_title(f'Total: {counts["total"]} runs')

# -- Overlap check --
ax2.axis('off')
overlap = qc.get('checks', {}).get('overlap', {})
if not overlap:
    overlap_text = '[PASS] No overlapping runs between any split pair.'
else:
    overlap_text = '[WARN] Overlap detected:\n' + str(overlap)

count_check = qc.get('checks', {}).get('count_match', {})
count_ok = count_check.get('expected') == count_check.get('actual')
count_text = '[PASS] Split counts match specification.' if count_ok else '[WARN] Count mismatch!'

ax2.text(0.1, 0.8, f'{count_text}\n\n{overlap_text}',
         transform=ax2.transAxes, fontsize=12, va='top',
         bbox=dict(boxstyle='round', facecolor='#d5f5e3', alpha=0.9))
ax2.set_title('Integrity checks')

plt.show()

## 2 - Categorical parameter balance

Discrete parameters should be approximately balanced across splits.

In [ ]:
cat_params = [
    ('patch_width', qc['patch_width_counts']),
    ('patch_offset', qc['patch_offset_counts']),
    ('k_dermis', qc['k_dermis_counts']),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
fig.suptitle('Categorical parameter distribution by split', fontsize=14, fontweight='bold')

split_colours = {'train': '#3498db', 'val': '#f39c12', 'test': '#e74c3c'}

for ax, (param_name, param_data) in zip(axes, cat_params):
    categories = sorted(param_data['train'].keys(), key=str)
    x = np.arange(len(categories))
    width = 0.25

    for i, split in enumerate(splits):
        vals = [param_data[split].get(c, 0) for c in categories]
        ax.bar(x + i * width, vals, width, label=split,
               color=split_colours[split], alpha=0.85)

    ax.set_xticks(x + width)
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_title(param_name)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.show()

## 3 - Continuous parameter distributions

We load the actual feature arrays and plot histograms for each continuous parameter.

In [ ]:
# Load feature arrays from the ML-ready dataset
train_data = np.load('data/processed/ml/train.npz')
val_data = np.load('data/processed/ml/val.npz')
test_data = np.load('data/processed/ml/test.npz')

feature_names = meta['feature_names']

# Select physically meaningful features (not engineered)
phys_features = ['C0', 'decay_rate', 'heterogeneity_sigma', 'heterogeneity_steps']
phys_idxs = [feature_names.index(f) for f in phys_features if f in feature_names]

fig, axes = plt.subplots(1, len(phys_idxs), figsize=(4 * len(phys_idxs), 4),
                          constrained_layout=True)
fig.suptitle('Continuous parameter distributions', fontsize=14, fontweight='bold')

for ax, fi in zip(axes, phys_idxs):
    fname = feature_names[fi]
    for data, label, c in [(train_data, 'train', '#3498db'),
                            (val_data, 'val', '#f39c12'),
                            (test_data, 'test', '#e74c3c')]:
        ax.hist(data['X'][:, fi], bins=20, alpha=0.5, color=c, label=label,
                edgecolor='white', linewidth=0.5)
    ax.set_title(fname)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

plt.show()

## 4 - Target (output) distributions

The five scalar targets span many orders of magnitude -- visualising their log-distributions
confirms the diversity needed for robust surrogate training.

In [ ]:
target_names = meta['scalar_target_names']

fig, axes = plt.subplots(1, len(target_names), figsize=(3.5 * len(target_names), 4),
                          constrained_layout=True)
fig.suptitle('Scalar target distributions (log_1_0 scale)', fontsize=14, fontweight='bold')

for ax, ti in zip(axes, range(len(target_names))):
    tname = target_names[ti]
    for data, label, c in [(train_data, 'train', '#3498db'),
                            (val_data, 'val', '#f39c12'),
                            (test_data, 'test', '#e74c3c')]:
        vals = data['y_scalar'][:, ti]
        # Use log10 for wide-range targets, linear for t_peak
        if tname == 't_peak':
            ax.hist(vals, bins=20, alpha=0.5, color=c, label=label,
                    edgecolor='white', linewidth=0.5)
        else:
            ax.hist(np.log10(np.abs(vals) + 1e-30), bins=20, alpha=0.5,
                    color=c, label=label, edgecolor='white', linewidth=0.5)
    ax.set_title(tname)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2)

plt.show()

## 5 - Flux curve diversity

A random sample of flux curves from the training set, coloured by `patch_width`,
shows the rich variety the surrogates must capture.

In [ ]:
pw_idx = feature_names.index('patch_width')
pw_map = {0.25: '#e74c3c', 0.5: '#2980b9', 1.0: '#27ae60'}

rng = np.random.default_rng(42)
sample_idxs = rng.choice(train_data['J'].shape[0], size=60, replace=False)

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.set_title('Training flux curves (sample of 60)', fontsize=13, fontweight='bold')

for i in sample_idxs:
    t_h = train_data['t'][i] / 3600.0
    J_i = train_data['J'][i]
    pw = train_data['X'][i, pw_idx]
    # Find closest key
    closest_pw = min(pw_map.keys(), key=lambda k: abs(k - pw))
    ax.plot(t_h, J_i, alpha=0.35, lw=0.8, color=pw_map[closest_pw])

# Legend
for pw_val, c in pw_map.items():
    ax.plot([], [], color=c, lw=2, label=f'patch_width = {pw_val}')
ax.legend(fontsize=10)
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Flux $J$')
ax.grid(alpha=0.3)
plt.show()

## 6 - Feature engineering summary

Beyond the 7 physical parameters, 13 engineered features capture interaction effects
and diffusivity statistics.

In [ ]:
phys = feature_names[:7]
eng = feature_names[7:]

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.axis('off')

table_data = []
for i, f in enumerate(feature_names):
    kind = 'Physical' if f in phys else 'Engineered'
    x_all = np.concatenate([train_data['X'][:, i], val_data['X'][:, i], test_data['X'][:, i]])
    table_data.append([f, kind, f'{x_all.min():.4e}', f'{x_all.max():.4e}', f'{x_all.mean():.4e}'])

table = ax.table(
    cellText=table_data,
    colLabels=['Feature', 'Type', 'Min', 'Max', 'Mean'],
    cellLoc='center', loc='center',
    colColours=['#d6eaf8'] * 5,
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.3)
ax.set_title('Feature summary (all 20 features)', fontsize=13, fontweight='bold', pad=20)

plt.show()

---

### Take-aways

- The dataset has **1,000 runs** with a clean 70/15/15 split and no overlaps.
- Categorical parameters are approximately balanced across splits.
- Continuous parameters cover their design ranges uniformly.
- Flux curves span a wide diversity of shapes, amplitudes, and peak times.
- 20 features (7 physical + 13 engineered) provide rich input representations.

**Next notebook ->** Surrogate model comparison (black-box vs PINN).